#### Imports / setup

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics

from mlxtend.plotting import plot_confusion_matrix
from mlxtend.plotting import plot_decision_regions

from matplotlib import pyplot as plt

from pathlib import Path

import tensorflow
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras import optimizers, regularizers
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback

print(tensorflow.__version__)


2.20.0


### Charger les données

In [2]:
df = pd.read_csv('./dataset/labels.csv')
df["filename"] = df["id"] + ".jpg"
train_dir = "dataset/train"
df = df[df["filename"].apply(lambda f: os.path.exists(os.path.join(train_dir, f)))].copy()
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["breed"], random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")
train_df.head()

Train: 8177, Val: 2045


,id,breed,filename
1926,2f812a0cb6977bbad1a167e1ef4ae2ac,gordon_setter,2f812a0cb6977bbad1a167e1ef4ae2ac.jpg
735,123c19c8d168e7704273cb7174351821,vizsla,123c19c8d168e7704273cb7174351821.jpg
2046,3303feb629eef7ee44f6398c91745f73,australian_terrier,3303feb629eef7ee44f6398c91745f73.jpg
3838,5f14fac852ee51524997243f086e4ea2,norwich_terrier,5f14fac852ee51524997243f086e4ea2.jpg
5252,8463aa43d88bee057082434ccc806bb0,bernese_mountain_dog,8463aa43d88bee057082434ccc806bb0.jpg


### Traitement de données

### Créer le modèle

In [ ]:
os.makedirs("model", exist_ok=True)

# Rotations, zooms, flips pour mieux généraliser
datagen_train = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode="nearest",
    brightness_range=[0.8, 1.2],
    channel_shift_range=20
)

datagen_val = ImageDataGenerator(preprocessing_function=mobilenet_preprocess)

train_gen = datagen_train.flow_from_dataframe(
    train_df, directory=train_dir, x_col="filename", y_col="breed",
    target_size=(224, 224), batch_size=32, class_mode="categorical", shuffle=True, seed=42,
)
val_gen = datagen_val.flow_from_dataframe(
    val_df, directory=train_dir, x_col="filename", y_col="breed",
    target_size=(224, 224), batch_size=32, class_mode="categorical", shuffle=False,
)

base = MobileNetV2(input_shape=(224, 224, 3), weights="imagenet", include_top=False, pooling=None)
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(1024, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

# x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

outputs = Dense(120, activation="softmax")(x)
model = Model(inputs=base.input, outputs=outputs, name="dog_breed_model")
model.compile(optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Found 8177 validated image filenames belonging to 120 classes.
Found 2045 validated image filenames belonging to 120 classes.


Model: "dog_breed_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 4,162,232 (15.88 MB)

 Trainable params: 1,901,176 (7.25 MB)

 Non-trainable params: 2,261,056 (8.63 MB)

### Callbacks d'entraînement

In [4]:
# Sauvegarde le meilleur modèle basé sur val_accuracy
model_checkpoint = ModelCheckpoint(
    "model/dog_breed_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)
callbacks = [model_checkpoint]

### Entraînement du modèle

In [5]:
EPOCHS = 80
STEPS = max(1, len(train_df) // 32)
VAL_STEPS = max(1, len(val_df) // 32)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    steps_per_epoch=STEPS,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/80
255/255 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.0563 - loss: 5.1657
Epoch 1: val_accuracy improved from None to 0.56101, saving model to model/dog_breed_best.keras

Epoch 1: finished saving model to model/dog_breed_best.keras
255/255 ━━━━━━━━━━━━━━━━━━━━ 48s 178ms/step - accuracy: 0.1228 - loss: 4.3900 - val_accuracy: 0.5610 - val_loss: 2.3187
Epoch 2/80
  1/255 ━━━━━━━━━━━━━━━━━━━━ 40s 160ms/step - accuracy: 0.1875 - loss: 3.5052

/Users/tomlau/Documents/ecole/dog-breed/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_accuracy improved from 0.56101 to 0.56349, saving model to model/dog_breed_best.keras

Epoch 2: finished saving model to model/dog_breed_best.keras
255/255 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.1875 - loss: 3.5052 - val_accuracy: 0.5635 - val_loss: 2.3102
Epoch 3/80
255/255 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.3420 - loss: 2.7645
Epoch 3: val_accuracy improved from 0.56349 to 0.68948, saving model to model/dog_breed_best.keras

Epoch 3: finished saving model to model/dog_breed_best.keras
255/255 ━━━━━━━━━━━━━━━━━━━━ 47s 186ms/step - accuracy: 0.3758 - loss: 2.5568 - val_accuracy: 0.6895 - val_loss: 1.2164
Epoch 4/80
  1/255 ━━━━━━━━━━━━━━━━━━━━ 33s 131ms/step - accuracy: 0.4375 - loss: 2.3296
Epoch 4: val_accuracy improved from 0.68948 to 0.69048, saving model to model/dog_breed_best.keras

Epoch 4: finished saving model to model/dog_breed_best.keras
255/255 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.4375 - loss: 2.3296 - val_accuracy: 0.6905 - 

KeyboardInterrupt: 

### Tests du modèle

In [6]:
val_gen.reset()
loss, accuracy = model.evaluate(val_gen, steps=VAL_STEPS, verbose=1)
print(f"Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

# S'assurer que model/labels.json a le même ordre que les classes du générateur
classes = [c for c, _ in sorted(train_gen.class_indices.items(), key=lambda x: x[1])]
with open("model/labels.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, indent=2, ensure_ascii=False)
print("Modèle et labels enregistrés.")

63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 116ms/step - accuracy: 0.7927 - loss: 0.6748
Validation Loss: 0.6748, Validation Accuracy: 0.7927
Modèle et labels enregistrés.
